## Libraries

In [ ]:
import wandb
from lightning.pytorch.loggers import WandbLogger
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint

In [ ]:
from utils.visualization import *
from utils.dataset import *
from utils.miscellaneous import *
from utils.scaling import *
from models.gnn import *
from models.models import *
from training.train import *
from training.loss import *
from database.graph_creation import *
from database.dhydro_utils import *

### Plot details

In [ ]:
import matplotlib as mpl

mpl.rcParams['grid.color'] = 'k'
mpl.rcParams['grid.linestyle'] = ':'
mpl.rcParams['grid.linewidth'] = 0.5

mpl.rcParams['figure.figsize'] = [7, 5]
mpl.rcParams['figure.dpi'] = 100
mpl.rcParams['savefig.dpi'] = 300
mpl.rcParams['savefig.bbox'] = 'tight'

mpl.rcParams['font.size'] = 18
mpl.rcParams['legend.fontsize'] = 'small'
mpl.rcParams['figure.titlesize'] = 'medium'

mpl.rcParams['font.family'] = 'serif'

figures_folder = 'results'

## Dataset creation

In [ ]:
# wandb.login()

cfg_file = "config_test.yaml"
config = read_config(cfg_file)

wandb.finish()

wandb.init(config=config, 
           project="mSWEGNN", 
           mode='disabled',
           )

wandb_logger = WandbLogger(log_model='all')

config = wandb.config

In [ ]:
torch.backends.cudnn.deterministic = True
torch.set_float32_matmul_precision('high')  # allows TF32 accumulation in matmul

L.seed_everything(config.models['seed'])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

dataset_parameters = config['dataset_parameters']
scalers = copy(config['scalers'])
selected_node_features = config['selected_node_features']
selected_edge_features = config['selected_edge_features']

train_dataset, val_dataset, test_dataset, scalers = create_model_dataset(
    scalers=scalers, **dataset_parameters, **selected_node_features, **selected_edge_features,
)

with_polygon_mesh = dataset_parameters.get('dataset_folder').split('/')[-1] == 'polygon_meshes'

test_graph_files = test_dataset.graph_files
train_dataset_graph_files = train_dataset.graph_files
val_dataset_graph_files = val_dataset.graph_files

In [ ]:
temporal_dataset_parameters = config['temporal_dataset_parameters']
# 
temporal_train_dataset = TemporalFloodDataset(train_dataset, **temporal_dataset_parameters)

if temporal_train_dataset.save_on_gpu:
    temporal_train_dataset = temporal_train_dataset._save_on_gpu()

# Model

## Creation

In [ ]:
temporal_res = dataset_parameters['temporal_res']
previous_t = temporal_dataset_parameters['previous_t']
future_t = temporal_dataset_parameters['future_t']
time_start = temporal_dataset_parameters['time_start']
time_stop = temporal_dataset_parameters['time_stop']
max_rollout_steps = temporal_dataset_parameters['rollout_steps']
test_dataset_name = dataset_parameters['test_dataset_name']

num_node_features = NUM_WATER_VARS*previous_t + sum(selected_node_features.values())
num_edge_features = sum(selected_edge_features.values())

print('Number of training simulations:\t', len(train_dataset))
print('Number of training samples:\t', len(temporal_train_dataset))
print('Number of node features:\t', num_node_features)
print('Number of rollout steps:\t', max_rollout_steps)

In [ ]:
model_parameters = copy(config['models'])
model_type = model_parameters.pop('model_type')

if model_type == 'MSGNN':
    num_scales = train_dataset[0].mesh.num_meshes
    model_parameters['num_scales'] = num_scales
    
model = get_model(model_type)(
    num_node_features=num_node_features,
    num_edge_features=num_edge_features,
    previous_t=previous_t,
    future_t=future_t,
    **model_parameters)

In [ ]:
trainer_options = copy(config['trainer_options'])
lr_info = config['lr_info']

# info for testing dataset
temporal_test_dataset_parameters = get_temporal_test_dataset_parameters(
    config, temporal_dataset_parameters)

# Validation dataset
mesh_file = os.path.join('results', "base_meshes.pkl") if with_polygon_mesh else os.path.join('results', "base_meshes_no_polygon.pkl")
if trainer_options.get('validation_method', 'normal') == 'mass_val':
    with open(mesh_file, 'rb') as f:
        base_datas = pickle.load(f)

    num_scenarios_per_location = config.get('num_scenarios_per_location', 2)

    num_breaches = len(base_datas)
    num_scenarios = num_breaches * num_scenarios_per_location

    # generate BCs
    peak_value=(200, 1400) # uniform distribution (change below)
    min_discharge=0
    param1=(1, 0.4)
    param2=(0., 0.2)
    param3=(5,15)
    param4=0.6
    x_window=(0.2, 1.6)

    max_time = test_dataset[0].WD.shape[1]

    all_hydrographs = []
    for i in range(num_scenarios):
        np.random.seed(i)

        hydrograph_time_series = generate_realistic_hydrograph(max_time-1, 1, 
                    np.random.uniform(*peak_value),
                    # np.linspace(peak_value[0], peak_value[1], num_scenarios_per_location)[i % num_scenarios_per_location]*np.random.uniform(0.7, 1.8),
                    # np.mean((peak_value[0],peak_value[1]))*np.random.uniform(0.4, 2),
                    min_discharge, 
                    np.random.lognormal(*param1), 
                    np.random.uniform(*param2), 
                    np.random.uniform(*param3), 
                    param4,
                    x_window)
        all_hydrographs.append(hydrograph_time_series[1])

    all_hydrographs = np.array(all_hydrographs)
    all_hydrographs[all_hydrographs < 1e-10] = 0

    temporal_val_dataset = create_prob_test_dataset(base_datas, all_hydrographs, for_execution=True, temporal_res=temporal_res, scalers=scalers,
                            **temporal_test_dataset_parameters, **selected_node_features, **selected_edge_features)
else:
    temporal_val_dataset = TemporalFloodDataset(val_dataset, rollout_steps=-1, **temporal_test_dataset_parameters)
    if temporal_dataset_parameters.get('save_on_gpu', False):
        temporal_val_dataset = temporal_val_dataset._save_on_gpu()

In [ ]:
pldatamodule = DataModule(temporal_train_dataset, temporal_val_dataset, 
                          batch_size=trainer_options['batch_size'])

plmodel = LightningTrainer(model, lr_info, trainer_options, temporal_test_dataset_parameters)

print("Total Number of paramters:", sum(p.numel() for p in model.parameters()))

## Training

In [ ]:
# Define callbacks
monitor_metric = "num_valid_simulations" if trainer_options.get('validation_method', 'normal') == 'mass_val' else "val_loss_CSI"
mode = 'min' if monitor_metric == "val_loss_CSI" else 'max'

checkpoint_callback = ModelCheckpoint(dirpath='lightning_logs/models', monitor=monitor_metric,
                                      mode=mode, save_top_k=2, save_last=True)
early_stopping = EarlyStopping(monitor_metric, mode=mode, patience=trainer_options['patience'])
curriculum_callback = CurriculumLearning(max_rollout_steps, mode=trainer_options['curriculum_mode'], patience=5)
# wandb_logger.watch(model, log="all", log_graph=False)

# Load trained model
plmodule_kwargs = {'model': model, 
                   'lr_info': lr_info, 
                   'trainer_options': trainer_options, 
                   'temporal_test_dataset_parameters': temporal_test_dataset_parameters}

if 'saved_model' in config:
  plmodel = LightningTrainer.load_from_checkpoint(config['saved_model'], **plmodule_kwargs)
  model = plmodel.model
  
num_GPUs = torch.cuda.device_count()

# Define trainer
trainer = L.Trainer(
                    accelerator="auto", 
                    devices=num_GPUs if num_GPUs > 0 else 'auto',
                    strategy='ddp_find_unused_parameters_true' if num_GPUs > 1 else 'auto',
                    max_epochs=trainer_options['max_epochs'],
                    gradient_clip_val=1, 
                    # log_every_n_steps=50,
                    # enable_progress_bar=False,
                    # profiler="simple",
                    precision='bf16-mixed',
                    # deterministic=True,
                    logger=wandb_logger,
                    callbacks=[checkpoint_callback, 
                               curriculum_callback, 
                               early_stopping, 
                               ])

In [ ]:
# # Train and get trained model
# trainer.fit(plmodel, pldatamodule, ckpt_path=config.get('saved_model', None))

# # Load the best model checkpoint
# plmodel = LightningTrainer.load_from_checkpoint(checkpoint_callback.best_model_path, **plmodule_kwargs)
# model = plmodel.model

# # validate with trained model
# trainer.validate(plmodel, pldatamodule)

## Test no edge

In [ ]:
num_random_tests = 1
num_scenarios = len(val_dataset) * num_random_tests
test_edge_dataset = []

temporal_train_dataset = TemporalFloodDataset(val_dataset, rollout_steps=-1, **temporal_test_dataset_parameters)

for i in range(num_scenarios):
    temp = copy(temporal_train_dataset[i % len(temporal_train_dataset)])
    weir_edges = torch.where(temp.edge_attr[:,1] != 0)[0]
    # selected_edges = weir_edges[np.random.choice(len(weir_edges), size=100, replace=False)]
    temp.edge_attr[:,1][weir_edges] = 0

    test_edge_dataset.append(temp)

# Train
train_dataloader = DataLoader(temporal_train_dataset, batch_size=2, shuffle=False)

start_time = time.time()
predicted_rollout = trainer.predict(plmodel, dataloaders=train_dataloader)
prediction_times = time.time() - start_time
# prediction_times = prediction_times/len(test_dataset)
predicted_rollout = [item for roll in predicted_rollout for item in roll]

spatial_analyser_train = SpatialAnalysis(predicted_rollout, prediction_times, val_dataset, **temporal_test_dataset_parameters)
spatial_analyser_train.get_plausible_runs()

# Train without edges
train_dataloader = DataLoader(test_edge_dataset, batch_size=2, shuffle=False)

start_time = time.time()
predicted_rollout = trainer.predict(plmodel, dataloaders=train_dataloader)
prediction_times = time.time() - start_time
# prediction_times = prediction_times/len(test_dataset)
predicted_rollout = [item for roll in predicted_rollout for item in roll]

spatial_analyser_train_no_edge = SpatialAnalysis(predicted_rollout, prediction_times, val_dataset, **temporal_test_dataset_parameters)
spatial_analyser_train_no_edge.get_plausible_runs()

In [ ]:
num_random_tests = 1
num_scenarios = len(test_dataset) * num_random_tests
test_edge_dataset = []

temporal_test_dataset = TemporalFloodDataset(test_dataset, rollout_steps=-1, **temporal_test_dataset_parameters)

for i in range(num_scenarios):
    temp = copy(temporal_test_dataset[i % len(temporal_test_dataset)])
    weir_edges = torch.where(temp.edge_attr[:,1] != 0)[0]
    # selected_edges = weir_edges[np.random.choice(len(weir_edges), size=100, replace=False)]
    temp.edge_attr[:,1][weir_edges] = 0

    test_edge_dataset.append(temp)

# Train
test_dataloader = DataLoader(temporal_test_dataset, batch_size=2, shuffle=False)

start_time = time.time()
predicted_rollout = trainer.predict(plmodel, dataloaders=test_dataloader)
prediction_times = time.time() - start_time
# prediction_times = prediction_times/len(test_dataset)
predicted_rollout = [item for roll in predicted_rollout for item in roll]

spatial_analyser_test = SpatialAnalysis(predicted_rollout, prediction_times, test_dataset, **temporal_test_dataset_parameters)
spatial_analyser_test.get_plausible_runs()

# Train without edges
test_dataloader = DataLoader(test_edge_dataset, batch_size=2, shuffle=False)

start_time = time.time()
predicted_rollout = trainer.predict(plmodel, dataloaders=test_dataloader)
prediction_times = time.time() - start_time
# prediction_times = prediction_times/len(test_dataset)
predicted_rollout = [item for roll in predicted_rollout for item in roll]

spatial_analyser_test_no_edge = SpatialAnalysis(predicted_rollout, prediction_times, test_dataset, **temporal_test_dataset_parameters)
spatial_analyser_test_no_edge.get_plausible_runs()

In [ ]:
# Train metrics
CSI005_train = spatial_analyser_train._get_CSI(water_threshold=0.05)
CSI03_train = spatial_analyser_train._get_CSI(water_threshold=0.3)
rollout_loss_train = spatial_analyser_train._get_rollout_loss(type_loss='MAE', only_where_water=False)

mae_h_train = rollout_loss_train[:, 0].mean().item() * 100
mae_h_train_std = rollout_loss_train[:, 0].std().item() * 100
mae_q_train = rollout_loss_train[:, 1].mean().item() * 100
mae_q_train_std = rollout_loss_train[:, 1].std().item() * 100
csi_005_train = CSI005_train.mean().item() * 100
csi_005_train_std = CSI005_train.std().item() * 100
csi_03_train = CSI03_train.mean().item() * 100
csi_03_train_std = CSI03_train.std().item() * 100

print(f"train & {mae_h_train:.2f} $\\pm$ {mae_h_train_std:.2f} & {mae_q_train:.2f} $\\pm$ {mae_q_train_std:.2f} & {csi_005_train:.2f} $\\pm$ {csi_005_train_std:.2f} & {csi_03_train:.2f} $\\pm$ {csi_03_train_std:.2f} \\\\")

# Train no edge metrics
CSI005_train_no_edge = spatial_analyser_train_no_edge._get_CSI(water_threshold=0.05)
CSI03_train_no_edge = spatial_analyser_train_no_edge._get_CSI(water_threshold=0.3)
rollout_loss_train_no_edge = spatial_analyser_train_no_edge._get_rollout_loss(type_loss='MAE', only_where_water=False)

mae_h_train_no_edge = rollout_loss_train_no_edge[:, 0].mean().item() * 100
mae_h_train_std_no_edge = rollout_loss_train_no_edge[:, 0].std().item() * 100
mae_q_train_no_edge = rollout_loss_train_no_edge[:, 1].mean().item() * 100
mae_q_train_std_no_edge = rollout_loss_train_no_edge[:, 1].std().item() * 100
csi_005_train_no_edge = CSI005_train_no_edge.mean().item() * 100
csi_005_train_std_no_edge = CSI005_train_no_edge.std().item() * 100
csi_03_train_no_edge = CSI03_train_no_edge.mean().item() * 100
csi_03_train_std_no_edge = CSI03_train_no_edge.std().item() * 100

print(f"train (edge = 0) & {mae_h_train_no_edge:.2f} $\\pm$ {mae_h_train_std_no_edge:.2f} & {mae_q_train_no_edge:.2f} $\\pm$ {mae_q_train_std_no_edge:.2f} & {csi_005_train_no_edge:.2f} $\\pm$ {csi_005_train_std_no_edge:.2f} & {csi_03_train_no_edge:.2f} $\\pm$ {csi_03_train_std_no_edge:.2f} \\\\")


# Test metrics
CSI005_train = spatial_analyser_test._get_CSI(water_threshold=0.05)
CSI03_train = spatial_analyser_test._get_CSI(water_threshold=0.3)
rollout_loss_train = spatial_analyser_test._get_rollout_loss(type_loss='MAE', only_where_water=False)

mae_h_train = rollout_loss_train[:, 0].mean().item() * 100
mae_h_train_std = rollout_loss_train[:, 0].std().item() * 100
mae_q_train = rollout_loss_train[:, 1].mean().item() * 100
mae_q_train_std = rollout_loss_train[:, 1].std().item() * 100
csi_005_train = CSI005_train.mean().item() * 100
csi_005_train_std = CSI005_train.std().item() * 100
csi_03_train = CSI03_train.mean().item() * 100
csi_03_train_std = CSI03_train.std().item() * 100

print(f"test & {mae_h_train:.2f} $\\pm$ {mae_h_train_std:.2f} & {mae_q_train:.2f} $\\pm$ {mae_q_train_std:.2f} & {csi_005_train:.2f} $\\pm$ {csi_005_train_std:.2f} & {csi_03_train:.2f} $\\pm$ {csi_03_train_std:.2f} \\\\")

# Train no edge metrics
CSI005_train_no_edge = spatial_analyser_test_no_edge._get_CSI(water_threshold=0.05)
CSI03_train_no_edge = spatial_analyser_test_no_edge._get_CSI(water_threshold=0.3)
rollout_loss_train_no_edge = spatial_analyser_test_no_edge._get_rollout_loss(type_loss='MAE', only_where_water=False)

mae_h_train_no_edge = rollout_loss_train_no_edge[:, 0].mean().item() * 100
mae_h_train_std_no_edge = rollout_loss_train_no_edge[:, 0].std().item() * 100
mae_q_train_no_edge = rollout_loss_train_no_edge[:, 1].mean().item() * 100
mae_q_train_std_no_edge = rollout_loss_train_no_edge[:, 1].std().item() * 100
csi_005_train_no_edge = CSI005_train_no_edge.mean().item() * 100
csi_005_train_std_no_edge = CSI005_train_no_edge.std().item() * 100
csi_03_train_no_edge = CSI03_train_no_edge.mean().item() * 100
csi_03_train_std_no_edge = CSI03_train_no_edge.std().item() * 100

print(f"test (edge = 0) & {mae_h_train_no_edge:.2f} $\\pm$ {mae_h_train_std_no_edge:.2f} & {mae_q_train_no_edge:.2f} $\\pm$ {mae_q_train_std_no_edge:.2f} & {csi_005_train_no_edge:.2f} $\\pm$ {csi_005_train_std_no_edge:.2f} & {csi_03_train_no_edge:.2f} $\\pm$ {csi_03_train_std_no_edge:.2f} \\\\")

In [ ]:
# val_dataloader = DataLoader(temporal_val_dataset, batch_size=trainer_options['batch_size'], shuffle=False, num_workers=0)
# trainer.validate(plmodel, dataloaders=val_dataloader)

In [ ]:
# trainer.fit(plmodel, pldatamodule)

# # Load the best model checkpoint
# plmodel = LightningTrainer.load_from_checkpoint(checkpoint_callback.best_model_path, **plmodule_kwargs)
# model = plmodel.model

## Testing

In [ ]:
# Train
temporal_train_dataset = TemporalFloodDataset(train_dataset, rollout_steps=-1, **temporal_test_dataset_parameters)
train_dataloader = DataLoader(temporal_train_dataset, batch_size=2, shuffle=False)

start_time = time.time()
predicted_rollout = trainer.predict(plmodel, dataloaders=train_dataloader)
prediction_times = time.time() - start_time
# prediction_times = prediction_times/len(test_dataset)
predicted_rollout = [item for roll in predicted_rollout for item in roll]

spatial_analyser_train = SpatialAnalysis(predicted_rollout, prediction_times, train_dataset, **temporal_test_dataset_parameters)
spatial_analyser_train.get_plausible_runs()

# Val
temporal_val_dataset = TemporalFloodDataset(val_dataset, rollout_steps=-1, **temporal_test_dataset_parameters)
val_dataloader = DataLoader(temporal_val_dataset, batch_size=2, shuffle=False)

start_time = time.time()
predicted_rollout = trainer.predict(plmodel, dataloaders=val_dataloader)
prediction_times = time.time() - start_time
# prediction_times = prediction_times/len(test_dataset)
predicted_rollout = [item for roll in predicted_rollout for item in roll]

spatial_analyser_val = SpatialAnalysis(predicted_rollout, prediction_times, val_dataset, **temporal_test_dataset_parameters)
spatial_analyser_val.get_plausible_runs()

# Test
temporal_test_dataset = TemporalFloodDataset(test_dataset, rollout_steps=-1, **temporal_test_dataset_parameters)
test_dataloader = DataLoader(temporal_test_dataset, batch_size=2, shuffle=False)

start_time = time.time()
predicted_rollout = trainer.predict(plmodel, dataloaders=test_dataloader)
prediction_times = time.time() - start_time
# prediction_times = prediction_times/len(test_dataset)
predicted_rollout = [item for roll in predicted_rollout for item in roll]

spatial_analyser_test = SpatialAnalysis(predicted_rollout, prediction_times, test_dataset, **temporal_test_dataset_parameters)
spatial_analyser_test.get_plausible_runs()

In [ ]:
print("MAE_h", "MAE_q", "CSI_005", "CSI_03")

# Train metrics
CSI005_train = spatial_analyser_train._get_CSI(water_threshold=0.05)
CSI03_train = spatial_analyser_train._get_CSI(water_threshold=0.3)
rollout_loss_train = spatial_analyser_train._get_rollout_loss(type_loss='MAE', only_where_water=False)

mae_h_train = rollout_loss_train[:, 0].mean().item() * 100
mae_h_train_std = rollout_loss_train[:, 0].std().item() * 100
mae_q_train = rollout_loss_train[:, 1].mean().item() * 100
mae_q_train_std = rollout_loss_train[:, 1].std().item() * 100
csi_005_train = CSI005_train.mean().item() * 100
csi_005_train_std = CSI005_train.std().item() * 100
csi_03_train = CSI03_train.mean().item() * 100
csi_03_train_std = CSI03_train.std().item() * 100

print(f"train & {mae_h_train:.2f} $\\pm$ {mae_h_train_std:.2f} & {mae_q_train:.2f} $\\pm$ {mae_q_train_std:.2f} & {csi_005_train:.2f} $\\pm$ {csi_005_train_std:.2f} & {csi_03_train:.2f} $\\pm$ {csi_03_train_std:.2f} \\\\")

# Val metrics
CSI005_val = spatial_analyser_val._get_CSI(water_threshold=0.05)
CSI03_val = spatial_analyser_val._get_CSI(water_threshold=0.3)
rollout_loss_val = spatial_analyser_val._get_rollout_loss(type_loss='MAE', only_where_water=False)

mae_h_val = rollout_loss_val[:, 0].mean().item() * 100
mae_h_val_std = rollout_loss_val[:, 0].std().item() * 100
mae_q_val = rollout_loss_val[:, 1].mean().item() * 100
mae_q_val_std = rollout_loss_val[:, 1].std().item() * 100
csi_005_val = CSI005_val.mean().item() * 100
csi_005_val_std = CSI005_val.std().item() * 100
csi_03_val = CSI03_val.mean().item() * 100
csi_03_val_std = CSI03_val.std().item() * 100

print(f"val & {mae_h_val:.2f} $\\pm$ {mae_h_val_std:.2f} & {mae_q_val:.2f} $\\pm$ {mae_q_val_std:.2f} & {csi_005_val:.2f} $\\pm$ {csi_005_val_std:.2f} & {csi_03_val:.2f} $\\pm$ {csi_03_val_std:.2f} \\\\")

# Test metrics
CSI005_test = spatial_analyser_test._get_CSI(water_threshold=0.05)
CSI03_test = spatial_analyser_test._get_CSI(water_threshold=0.3)
rollout_loss_test = spatial_analyser_test._get_rollout_loss(type_loss='MAE', only_where_water=False)

mae_h_test = rollout_loss_test[:, 0].mean().item() * 100
mae_h_test_std = rollout_loss_test[:, 0].std().item() * 100
mae_q_test = rollout_loss_test[:, 1].mean().item() * 100
mae_q_test_std = rollout_loss_test[:, 1].std().item() * 100
csi_005_test = CSI005_test.mean().item() * 100
csi_005_test_std = CSI005_test.std().item() * 100
csi_03_test = CSI03_test.mean().item() * 100
csi_03_test_std = CSI03_test.std().item() * 100

print(f"test & {mae_h_test:.2f} $\\pm$ {mae_h_test_std:.2f} & {mae_q_test:.2f} $\\pm$ {mae_q_test_std:.2f} & {csi_005_test:.2f} $\\pm$ {csi_005_test_std:.2f} & {csi_03_test:.2f} $\\pm$ {csi_03_test_std:.2f}\\\\")

In [ ]:
num_samples = len(test_dataset)
group_size = 3

# Prepare LaTeX table header
print(r"\begin{tabular}{lccc|ccccc}")
print(r"\textbf{dataset\_name} & \textbf{ID\_location} & \textbf{return period}& \textbf{Total flood volume $10^6 m^3$} & \textbf{MAE\_h} & \textbf{MAE\_q} & \textbf{CSI\_005} & \textbf{CSI\_03}\\")
print(r"\hline")

for i in range(num_samples):
    # Group ID_location and return period in 3s
    ID_location = ['ND234', 'HD073', 'ND111'][i // group_size]
    return_period = [10000,1000,100][i % group_size]

    # Get metrics for this sample
    spatial_analyser_test.get_plausible_runs()
    total_volume = spatial_analyser_test.real_volumes[i,-1]/1e6
    mae_h = round(rollout_loss_test[i, 0].item() * 100, 2)
    mae_q = round(rollout_loss_test[i, 1].item() * 100, 2)
    csi_005 = round(CSI005_test[i].mean().item() * 100, 2)
    csi_03 = round(CSI03_test[i].mean().item() * 100, 2)

    dataset_name = '' if i!=0 else '\\multirow{9}{*}{Test}'
    ID_location = '' if (i % group_size)!=0 else '\\multirow{3}{*}'f'{{{ID_location}}}'

    print(f"{dataset_name} & {ID_location} & {return_period} & {total_volume:.2f} & {mae_h} & {mae_q} & {csi_005} & {csi_03} \\\\")

print(r"\end{tabular}")

atomic-music-2404 (all)
train & 29.56 $\pm$ 17.23 & 1.46 $\pm$ 1.00 & 81.74 $\pm$ 9.34 & 80.40 $\pm$ 11.39 \\
val & 23.98 $\pm$ 12.64 & 1.41 $\pm$ 1.72 & 73.80 $\pm$ 13.54 & 71.54 $\pm$ 14.32 \\
test & 64.84 $\pm$ 81.38 & 1.31 $\pm$ 1.14 & 73.60 $\pm$ 14.90 & 71.15 $\pm$ 14.21\\

solar-universe-2437 (no node)
train & 26.57 $\pm$ 21.93 & 0.95 $\pm$ 0.76 & 83.95 $\pm$ 10.23 & 83.46 $\pm$ 11.90 \\
val & 27.45 $\pm$ 19.74 & 0.92 $\pm$ 0.94 & 72.31 $\pm$ 15.20 & 70.11 $\pm$ 15.46 \\
test & 70.83 $\pm$ 85.70 & 0.97 $\pm$ 0.87 & 73.58 $\pm$ 14.24 & 70.15 $\pm$ 13.70\\

splendid-lake-2428 (no polygon)
train & 37.98 $\pm$ 21.62 & 1.35 $\pm$ 0.92 & 76.48 $\pm$ 12.80 & 74.27 $\pm$ 15.23 \\
val & 36.45 $\pm$ 17.69 & 1.21 $\pm$ 1.16 & 68.22 $\pm$ 18.85 & 65.92 $\pm$ 19.16 \\
test & 69.06 $\pm$ 70.22 & 1.41 $\pm$ 1.01 & 68.72 $\pm$ 19.94 & 66.20 $\pm$ 19.12\\

daily-salad-2420 (only polygon)
train & 31.30 $\pm$ 19.18 & 1.23 $\pm$ 0.90 & 79.64 $\pm$ 11.88 & 79.08 $\pm$ 13.05 \\
val & 34.43 $\pm$ 22.04 & 1.22 $\pm$ 1.64 & 70.31 $\pm$ 16.03 & 67.97 $\pm$ 16.14 \\
test & 71.32 $\pm$ 77.96 & 1.28 $\pm$ 1.00 & 70.42 $\pm$ 19.01 & 67.72 $\pm$ 18.26\\

silvery-spaceship-2412 (no edge)
train & 29.87 $\pm$ 24.01 & 1.08 $\pm$ 0.83 & 80.63 $\pm$ 9.55 & 78.11 $\pm$ 12.31 \\
val & 36.71 $\pm$ 31.31 & 1.09 $\pm$ 1.17 & 69.14 $\pm$ 14.22 & 65.65 $\pm$ 15.60 \\
test & 68.99 $\pm$ 82.90 & 0.99 $\pm$ 0.88 & 73.27 $\pm$ 9.89 & 67.92 $\pm$ 10.26\\

swift-sound-2551 (no hydraulic structures)
train & 40.19 $\pm$ 27.03 & 1.24 $\pm$ 1.00 & 73.24 $\pm$ 11.85 & 69.43 $\pm$ 14.22 \\
val & 46.92 $\pm$ 28.70 & 1.12 $\pm$ 1.18 & 63.82 $\pm$ 17.66 & 60.00 $\pm$ 18.35 \\
test & 77.91 $\pm$ 79.46 & 1.20 $\pm$ 0.95 & 67.06 $\pm$ 18.91 & 62.53 $\pm$ 18.02\\

atomic-music-2404 (all - edge_weir = 0)
val & 60.48 $\pm$ 26.82 & 2.33 $\pm$ 1.41 & 60.75 $\pm$ 18.70 & 58.43 $\pm$ 20.36 \\
test & 85.30 $\pm$ 41.87 & 2.88 $\pm$ 1.42 & 63.49 $\pm$ 21.88 & 61.31 $\pm$ 21.32 \\

# Plots

## Exploratory analysis (single simulation)
Find the best and worst simulations in a given dataset

Then, you can plot the simulation summaries

In [ ]:
sorted_ids = spatial_analyser_test.plot_loss_per_simulation(type_loss='MAE', ranking='combined', only_where_water=False, water_thresholds=[0.05, 0.3, 1])

In [ ]:
# Prepare arrays for each split
ARME_train = np.asarray(spatial_analyser_train.ARME)
ARME_val = np.asarray(spatial_analyser_val.ARME)
ARME_test = np.asarray(spatial_analyser_test.ARME)

CSI_train_005 = spatial_analyser_train._get_CSI(0.05).nanmean(1).numpy()
CSI_train_03 = spatial_analyser_train._get_CSI(0.3).nanmean(1).numpy()

CSIs_val_005 = spatial_analyser_val._get_CSI(0.05).nanmean(1).numpy()
CSIs_val_03 = spatial_analyser_val._get_CSI(0.3).nanmean(1).numpy()

CSIs_test_005 = spatial_analyser_test._get_CSI(0.05).nanmean(1).numpy()
CSIs_test_03 = spatial_analyser_test._get_CSI(0.3).nanmean(1).numpy()

mae_h_train = rollout_loss_train[:, 0]
mae_q_train = rollout_loss_train[:, 1]

mae_h_val = rollout_loss_val[:, 0]
mae_q_val = rollout_loss_val[:, 1]

mae_h_test = rollout_loss_test[:, 0]
mae_q_test = rollout_loss_test[:, 1]

# === PART 1: GLOBAL CORRELATION ANALYSIS ===
summarize_correlation("train", ARME_train, CSI_train_005, CSI_train_03, mae_h_train, mae_q_train)
summarize_correlation("val", ARME_val, CSIs_val_005, CSIs_val_03, mae_h_val, mae_q_val)
summarize_correlation("test", ARME_test, CSIs_test_005, CSIs_test_03, mae_h_test, mae_q_test)

## Run single simulation

In [ ]:
id_dataset = 3

test_dataset.graph_files = [test_graph_files[id_dataset]]
# train_dataset.graph_files = [train_dataset_graph_files[id_dataset]]
# val_dataset.graph_files = [val_dataset_graph_files[id_dataset]]

rollout_plotter = PlotRollout(plmodel, trainer, test_dataset, 
                              scalers=scalers, type_loss='RMSE', **temporal_test_dataset_parameters)

rollout_plotter.plot_BC();

### Single spatial analysis

In [ ]:
rollout_plotter._plot_metric('CSI', water_thresholds=[0.05, 0.3, 1])
rollout_plotter._get_rollout_loss('MAE', only_where_water=False)

In [ ]:
rollout_plotter._plot_volumes()
rollout_plotter.get_volumes_r2(), rollout_plotter.get_volumes_ARME()

In [ ]:
# fig = rollout_plotter.explore_rollout(time_step=-1, scale=1, logscale=True, 
#                                     #   zoom_extent=rollout_plotter.around_breach
#                                       )
# # fig = rollout_plotter.explore_multiscale_rollout(time_step=-1, variable='V', logscale=True)

# import cv2, PIL
# fig.savefig("results/temp_summary.png")
# img = cv2.imread("results/temp_summary.png")
# image = wandb.Image(PIL.Image.fromarray(img), caption="Summary image")

# wandb.log({"summary image": image})
# plt.savefig(f'{figures_folder}/summmary_test_{id_dataset:02d}.png')

### Plot WD and V for a single simulation

In [ ]:
plot_times = [0,1,2,3,4]
plot_times = [5, 10, 20, 30]
# plot_times = [11, 23, 35, 47]

rollout_plotter.mesh_scale_plot(scale=1)

rollout_plotter.real_WD.kwargs['vmax'] = 2
rollout_plotter.predicted_WD.kwargs['vmax'] = 2
rollout_plotter.difference_WD.kwargs['vmax'] = 1
rollout_plotter.difference_WD.kwargs['vmin'] = -1

rollout_plotter.real_V.kwargs['vmax'] = 0.5
rollout_plotter.predicted_V.kwargs['vmax'] = 0.5
rollout_plotter.difference_V.kwargs['vmax'] = 0.1
rollout_plotter.difference_V.kwargs['vmin'] = -0.1

rollout_plotter.compare_h_rollout(plot_times, scale=None)

# rollout_plotter.compare_v_rollout(plot_times, scale=None, logscale=True)

# plt.savefig(f'{figures_folder}/pred_true_q_{id_dataset:02d}.png')
# plt.savefig(f'{figures_folder}/pred_true_h_{id_dataset:02d}.png')

### Compare flood arrival times (FAT)

In [ ]:
rollout_plotter.mesh_scale_plot(scale=0)

rollout_plotter.diff_FATPlot.kwargs['vmin'] = -96
rollout_plotter.diff_FATPlot.kwargs['vmax'] = 96
# rollout_plotter.pred_FATPlot.kwargs['vmax'] = 36
# rollout_plotter.real_FATPlot.kwargs['vmax'] = 36

rollout_plotter.compare_FAT(water_threshold=0.05, scale=None)
# plt.savefig(f'{figures_folder}/FAT_005_train_{id_dataset:02d}.png')

### Video

In [ ]:
# rollout_plotter.mesh_scale_plot(scale=0)

# rollout_plotter.real_WD.kwargs['vmax'] = 2
# rollout_plotter.predicted_WD.kwargs['vmax'] = 2
# rollout_plotter.difference_WD.kwargs['vmax'] = 1
# rollout_plotter.difference_WD.kwargs['vmin'] = -1

# rollout_plotter.real_V.kwargs['vmax'] = 0.5
# rollout_plotter.predicted_V.kwargs['vmax'] = 0.5
# rollout_plotter.difference_V.kwargs['vmax'] = 0.2
# rollout_plotter.difference_V.kwargs['vmin'] = -0.2

# rollout_plotter.create_video(logscale=True)
# # rollout_plotter.create_multiscale_video()
# rollout_plotter.save_video(f'results/dk43_test_{id_dataset:02d}', fps=4)

## Analysis on multiple simulations

### Boundary conditions

In [ ]:
plausible_runs = spatial_analyser_val.get_plausible_runs(ARME_threshold=0.4)
spatial_analyser_val._plot_BCs(highlight_ids=plausible_runs)
print(", ".join([f"{score:.2f}" for score in spatial_analyser_val.ARME]))
# plt.savefig(os.path.join(figures_folder, 'valid_BCs_train.png'), dpi=300, bbox_inches='tight')

In [ ]:
train_discharges = torch.cat([data.BC[data.type_BC == 2] * data.edge_BC_length[data.type_BC == 2] for data in train_dataset]).cpu().numpy()
val_discharges = torch.cat([data.BC[data.type_BC == 2] * data.edge_BC_length[data.type_BC == 2] for data in val_dataset]).cpu().numpy()
test_dataset.graph_files = test_graph_files
test_discharges = torch.cat([data.BC[data.type_BC == 2] * data.edge_BC_length[data.type_BC == 2] for data in test_dataset]).cpu().numpy()

train_breach_coords = np.stack([data.mesh.face_xy[data.node_BC] for data in train_dataset])
val_breach_coords = np.stack([data.mesh.face_xy[data.node_BC] for data in val_dataset])
test_breach_coords = np.stack([data.mesh.face_xy[data.node_BC] for data in test_dataset])

dict_breach_coords_discharges = {"Training": [train_breach_coords, train_discharges.max(1)], 
                              "Validation": [val_breach_coords, val_discharges.max(1)], 
                              "Testing": [test_breach_coords, None]
                              }

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(20, 6), width_ratios=[2,1])
gdf_mesh = train_dataset[0].mesh.meshes[-1]

# plot_breach_distribution(spatial_analyser_val.breach_coords, val_discharges, gdf_mesh, ax=axs[0], with_label=False, discharge_intervals=[500,1000], volume_intervals=[1.5e8, 2.5e8], 
#                          temporal_res=temporal_res, edgecolors='k', linewidths=0.5)
axs[0] = plot_discharge_breach_distribution(dict_breach_coords_discharges, gdf_mesh, ax=axs[0], cmap='cividis', edgecolor='white', linewidth=1)
# axs[0].set_aspect('equal')

# add text for text breach locations ND234, HD073, ND111
for i, (x, y) in enumerate(test_breach_coords):
    label = 'ND234' if i < 3 else 'HD073' if i < 6 else 'ND111'
    axs[0].text(x+500, y+550, label, fontsize=16, ha='center', va='bottom', color='k')
axs[0].scatter(151000, 436000, c='None')

time_vector = rollout_plotter.time_vector/24

plot_line_with_deviation(time_vector, train_discharges, with_minmax=True, c='C0', label='Training', ax=axs[1])
plot_line_with_deviation(time_vector, val_discharges, with_minmax=True, c='C1', label='Validation', ax=axs[1])

axs[1].plot(time_vector, test_discharges[:3].T, c='C2', label='ND234')
axs[1].plot(time_vector, test_discharges[3:6].T, c='C3', label='HD073')
axs[1].plot(time_vector, test_discharges[6:].T, c='C4', label='ND111')

axs[1].set_xlabel('Time [days]')
axs[1].set_ylabel('Discharge [m$^3$/s]')

# Only show one legend entry per label
handles, labels = plt.gca().get_legend_handles_labels()
by_label = dict(zip(labels, handles))
plt.legend(by_label.values(), by_label.keys())

plt.tight_layout()

axs[0].text(0.99, 0.98, '(a)', transform=axs[0].transAxes,
            fontsize=20, va='top', ha='right')    
axs[1].text(0.02, 0.98, '(b)', transform=axs[1].transAxes,
            fontsize=20, va='top', ha='left');

plt.savefig(f'{figures_folder}/discharges_distributions.png', dpi=300, bbox_inches='tight')

### Spatial metrics (F1 and CSI)

In [ ]:
mpl.rcParams['font.size'] = 18

fig, axs = plt.subplots(1, 2, figsize=(12, 5))

_, CSI = spatial_analyser_test.plot_CSI_rollouts(water_thresholds=[0.05, 0.3], ax=axs[0])
print(np.nanmean(CSI, 1).mean(1))
# _, F1 = spatial_analyser.plot_F1_rollouts(water_thresholds=[0.05, 0.3], ax=axs[0])
# print(np.nanmean(F1, 1).mean(1))
# _ = spatial_analyser._plot_rollouts(type_loss='RMSE', ax=axs[1])
_ = spatial_analyser_test._plot_rollouts(type_loss='MAE', ax=axs[1])
axs[0].grid(False)

plt.tight_layout()

# plt.savefig(f'{figures_folder}/CSI_MAE_test.png')

# Probabilistic stuff

### Setup case study and mesh

In [ ]:
save_folder = config.get('save_folder', 'results')
# save_folder = os.path.join(save_folder, 'dk43')
os.makedirs(save_folder, exist_ok=True)

breach_id = config.get('breach_location', 'all')
return_period = config.get('return_period', None)
return_period = 10000
breach_id = 'HD073'

save_folder = os.path.join(config.get('save_folder', 'results'), breach_id)
if return_period is not None:
    save_folder = os.path.join(save_folder, f'RP{return_period}')
os.makedirs(save_folder, exist_ok=True)

time_stop = 64
type_BC = 2

case_study = 'dk41'
dataset_folder = f"database/raw_datasets_{case_study}"

dijkpalen_file = os.path.join(dataset_folder, "dijkpalen.gpkg")
dijkpalen = gpd.read_file(dijkpalen_file)
dijkpalen.sort_values(by='CODE', inplace=True)
dijkpalen = dijkpalen[dijkpalen.WS_DIJKRIN == 'DR41']
dijkpalen_coords = np.array([[geom.x, geom.y] for geom in dijkpalen.geometry])

if breach_id != 'all':
    if isinstance(breach_id, list):
        breach_index = np.where(dijkpalen.CODE.isin([b + '.' for b in breach_id]))[0]
    else:
        breach_index = np.where(dijkpalen.CODE == (breach_id + '.'))[0]
    BC_loc = np.array([(geom.x, geom.y) for geom in dijkpalen.iloc[breach_index].geometry])
else:
    selected_dijkpalen_HD = dijkpalen[dijkpalen.CODE.str.startswith(('HD'))][::22]
    selected_dijkpalen_ND = dijkpalen[dijkpalen.CODE.str.startswith(('ND'))][::22][::-1]

    selected_dijkpalen = pd.concat([selected_dijkpalen_HD, selected_dijkpalen_ND])
    # dijkpalen = dijkpalen[dijkpalen.CODE.str.startswith(('HD'))][::8]
    BC_loc = np.array([(geom.x, geom.y) for geom in selected_dijkpalen.geometry])

ID_locations = ['ND234', 'HD073', 'ND111']
return_periods = [10000, 1000, 100]
id_return_period_to_index = {(ID_locations[i // 3], return_periods[i % 3]): i for i in range(len(test_dataset))}
test_dataset_id = id_return_period_to_index.get((breach_id, return_period), None)

# point_id = 2
# BC_loc = points[[point_id]]
# BC_loc = portion[::5]

In [ ]:
plt.figure(figsize=(10, 5))

polygon_file = os.path.join(dataset_folder, "Geometry", f"boundary_{case_study}.pol")
boundary_polygon = Polygon(np.loadtxt(polygon_file))
boundary_coords = np.array(boundary_polygon.boundary.coords)
plt.plot(*boundary_coords.T)
plt.gca().set_aspect('equal')
correct_plt_units(plt.gca(), boundary_coords)

# plt.xticks([])
# plt.yticks([])

# plot points
print("Number of points: ", len(BC_loc))
plt.scatter(BC_loc[:,0], BC_loc[:,1], s=100, c='r', marker='x', zorder=2);
# for i, txt in enumerate(range(len(BC_loc))):
#     plt.annotate(txt, (BC_loc[i,0], BC_loc[i,1]))

# plt.savefig(os.path.join(figures_folder, 'breach_locations.png'), dpi=300, bbox_inches='tight')

In [ ]:
new_mesh_file = os.path.join(save_folder, "base_meshes.pkl")
mesh_file = os.path.join(dataset_folder, "base_meshes.pkl")

if not os.path.exists(new_mesh_file):
    base_datas = []
    for bc_loc in tqdm(BC_loc, total=len(BC_loc)):
        j = 1
        while True:
            with open(mesh_file, 'rb') as f:
                meshes = pickle.load(f)
                if with_polygon_mesh:
                    meshes.pop(2)
                    meshes.pop(2)
                    meshes.pop(2)
                else:
                    meshes.pop(0)
                    meshes.pop(0)
            base_data = add_BC_to_data(copy(meshes), np.array([bc_loc]), np.zeros(time_stop), type_BC=2)
            if base_data:
                _, closest_idx = cKDTree(dijkpalen_coords).query(bc_loc)
                base_data.CODE = dijkpalen.iloc[closest_idx].CODE
                base_datas.append(base_data)
                break
            else:
                _, closest_idx = cKDTree(dijkpalen_coords).query(bc_loc, k=j+1)
                closest_dijkpalen = dijkpalen.iloc[closest_idx[j:]]
                bc_loc = np.array([(geom.x, geom.y) for geom in closest_dijkpalen.geometry]).squeeze()
                j += 1

    mesh_counts = np.unique_counts([data.DEM.shape for data in base_datas])
    best_cell_num = mesh_counts[0][mesh_counts[1].argmax()]

    base_datas = [data for data in base_datas if data.DEM.shape[0] == best_cell_num]
        
    with open(new_mesh_file, 'wb') as f:
        pickle.dump(base_datas, f)
else:
    with open(new_mesh_file, 'rb') as f:
        base_datas = pickle.load(f)
    
with open(mesh_file, 'rb') as f:
    meshes = pickle.load(f)
    if with_polygon_mesh:
        meshes.pop(2)
        meshes.pop(2)
        meshes.pop(2)
    else:
        meshes.pop(0)
        meshes.pop(0)

num_breaches = len(base_datas)
print(f'Number of breach locations: {num_breaches}')

In [ ]:
if return_period is not None and breach_id != 'all':
   test_BC = pd.read_csv(os.path.join('database', "raw_datasets_dk41", "Probabilistic breach outflow", "processed", f"{breach_id}_T{return_period}.csv"), header=None)
   test_hydrograph = (test_dataset[test_dataset_id].BC * test_dataset[test_dataset_id].edge_BC_length)
   all_hydrographs = test_BC.values[:,2:]

   num_scenarios_per_location = len(all_hydrographs)
   num_scenarios = num_breaches * num_scenarios_per_location
   plt.plot(all_hydrographs.T)
else:
   num_scenarios_per_location = config.get('num_scenarios_per_location', 20)
   num_scenarios = num_breaches * num_scenarios_per_location

   # generate BCs
   peak_value=(6.2, 0.4) # loc, scale of the lognormal distribution
   peak_value=(100, 1500) # uniform distribution (change below)
   min_discharge=0
   param1=(1, 0.4)
   param2=(0., 0.2)
   param3=(5,15)
   param4=0.6
   x_window=(0.2, 1.6)

   all_hydrographs = []
   for i in range(num_scenarios):
      np.random.seed(i)

      hydrograph_time_series = generate_realistic_hydrograph(time_stop-1, 1, 
                                                            np.random.uniform(*peak_value),
                                                            min_discharge, 
                                                            np.random.lognormal(*param1), 
                                                            np.random.uniform(*param2), 
                                                            np.random.uniform(*param3), 
                                                            param4,
                                                            x_window)
      all_hydrographs.append(hydrograph_time_series[1])
      plt.plot(hydrograph_time_series[1])

   all_hydrographs = np.array(all_hydrographs)
   all_hydrographs[all_hydrographs < 1e-10] = 0
   # plt.savefig(os.path.join(save_folder, 'hydrographs2.png'))

### Do predictions

In [ ]:
prediction_file = os.path.join(save_folder, f"predictions46.npy")
volume_file = os.path.join(save_folder, f"volumes46.npy")

if not os.path.exists(prediction_file):
    prob_test_dataset = create_prob_test_dataset(base_datas, all_hydrographs, for_execution=True, temporal_res=temporal_res, scalers=scalers,
                            **temporal_test_dataset_parameters, **selected_node_features, **selected_edge_features)

    prob_test_dataloader = DataLoader(prob_test_dataset, batch_size=3, shuffle=False)

    start_time = time.time()
    predicted_rollout = trainer.predict(plmodel, dataloaders=prob_test_dataloader)
    prediction_times = time.time() - start_time
    predicted_rollout = [item for roll in predicted_rollout for item in roll]

    ensemble_selected_prediction = stack_rollout_different_BC(predicted_rollout, prob_test_dataset, scale=0)
    ensemble_predicted_volumes = torch.einsum('snt,n->st', ensemble_selected_prediction[:, :, 0].to(torch.float32), 
                                            torch.as_tensor(meshes[-1].face_area, device=ensemble_selected_prediction.device, dtype=torch.float32)).cpu().numpy()
    
    FAT_005 = torch.stack([WD_to_FAT(data[:, 0], temporal_res, 0.05, time_start) for data in ensemble_selected_prediction])
    FAT_03 = torch.stack([WD_to_FAT(data[:, 0], temporal_res, 0.3, time_start) for data in ensemble_selected_prediction])
    WD_max = torch.stack([data.max(2).values[:, 0] for data in ensemble_selected_prediction])
    V_max = torch.stack([data.max(2).values[:, 1] for data in ensemble_selected_prediction])
    WD_end = torch.stack([data[:, 0, -1] for data in ensemble_selected_prediction])

    ensemble_selected_prediction = torch.stack([FAT_005, FAT_03, WD_max, V_max, WD_end], dim=2).cpu().numpy()

    np.save(prediction_file, ensemble_selected_prediction)
    np.save(volume_file, ensemble_predicted_volumes)

else:
    ensemble_selected_prediction = np.load(prediction_file, mmap_mode='r')
    ensemble_predicted_volumes = np.load(volume_file, mmap_mode='r')

In [ ]:
prob_test_dataset = create_prob_test_dataset(base_datas, all_hydrographs, for_execution=True, temporal_res=temporal_res, scalers=scalers,
                         **temporal_test_dataset_parameters, **selected_node_features, **selected_edge_features)

### Summary all breaches

In [ ]:
base_DEM = base_datas[0].DEM if hasattr(base_datas[0], 'DEM') else None
base_mesh = meshes[-1]
gdf_mesh = meshes[0]

ARME_threshold = trainer_options.get("ARME_threshold", 0.4)

In [ ]:
fig = plot_breach_distribution_and_quantiles(ensemble_predicted_volumes, prob_test_dataset, num_breach_groups=8, 
                                                q_ranges=[0, 25, 50, 75, 100], max_ARME=2, show_percentage_runs=True, time_start=time_start)
# # plt.savefig(os.path.join(save_folder, 'breach_distribution_quantiles.png'), dpi=300, bbox_inches='tight')

In [ ]:
full_prob_analyser = ProbabilisticSpatialAnalysis(ensemble_selected_prediction, ensemble_predicted_volumes, prob_test_dataset,
                                                  scalers, base_DEM, base_mesh, **temporal_test_dataset_parameters)
plausible_runs = full_prob_analyser.get_plausible_runs(ARME_threshold=10)

In [ ]:
train_coords = np.stack([data.mesh.face_xy[data.node_BC] for data in train_dataset])
train_discharges = np.concatenate([data.BC[data.type_BC == 2] * data.edge_BC_length[data.type_BC == 2] for data in train_dataset])

fig = analyze_training_vs_good_tests(train_discharges, train_coords, all_hydrographs[:,:time_stop], BC_loc,
                                   full_prob_analyser.ARME, gdf_mesh, ARME_threshold=ARME_threshold, check_bads=True, k_neighbors=3)

In [ ]:
ax = plot_percentage_plausible_volumes_vs_ARME(full_prob_analyser.ARME, prob_test_dataset, ARME_thresholds=[1.0, 0.8, 0.6, 0.4, 0.2])

In [ ]:
fig, axs = plot_ARME_thresholds_analysis(full_prob_analyser, ensemble_selected_prediction, ensemble_predicted_volumes, prob_test_dataset, scalers, 
                                    temporal_test_dataset_parameters, ARME_thresholds=[0.4, 0.6, 1, 2])
# plt.savefig(os.path.join(save_folder, 'ARME_thresholds_analysis.png'), dpi=300, bbox_inches='tight')

In [ ]:
mask_volume = full_prob_analyser.ARME < ARME_threshold

selected_prob_dataset = [prob_test_dataset[i] for i in np.where(mask_volume)[0]]
prob_analyser = ProbabilisticSpatialAnalysis(ensemble_selected_prediction[mask_volume], ensemble_predicted_volumes[mask_volume],
                                             selected_prob_dataset, scalers, base_DEM, base_mesh, **temporal_test_dataset_parameters)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(20, 6), gridspec_kw={'width_ratios': [1, 2]})

full_prob_analyser.plot_runs_per_threshold(with_hist=True, ARME_thresholds=np.linspace(0, max(ARME_threshold, 1.5), 50), ax=axs[0],
                                            bins=40, alpha=0.4, color='b', edgecolor='k', linewidth=0.5)
# axs[0].axvline(x=0.2, color='g', linestyle='--', linewidth=1.5)
axs[0].axvline(x=ARME_threshold, color='k', linestyle='--', linewidth=1.5)
# axs[0].axvline(x=1, color='r', linestyle='--', linewidth=1.5)

plausible_runs = full_prob_analyser.get_plausible_runs(ARME_threshold=ARME_threshold)
plausible_runs_per_location = np.bincount(plausible_runs // num_scenarios_per_location, minlength=num_breaches)
plot_valid_runs_breach_distribution(base_datas, plausible_runs_per_location/num_scenarios_per_location*100, cmap='YlGn', edgecolor='k', linewidth=0.3, ax=axs[1])
axs[1].set_aspect('auto')

panel_labels = [f'({chr(97+i)})' for i in range(len(axs))]
for i, label in enumerate(panel_labels):
    axs[i].text(0.02, 0.98, label, transform=axs[i].transAxes,
                fontsize=20, va='top', ha='left')
    
plt.tight_layout()

In [ ]:
fig, axs = plot_percentiles(ensemble_selected_prediction, quantiles=[0.25, 0.5, 0.75], variable='V_max', 
                                    temporal_res=temporal_res, **prob_analyser.default_plot_kwargs)

In [ ]:
ax = prob_analyser.plot_volume_distribution(log_scale=False, 
                                            # time_step=-1, 
                                            );
# plt.savefig(f"{save_folder}/volume_distribution.png")
# plt.savefig(f"{save_folder}/selected_volume_distribution.png")

In [ ]:
mask_volume = full_prob_analyser.ARME >= ARME_threshold

anti_selected_prob_dataset = [prob_test_dataset[i] for i in np.where(mask_volume)[0]]
anti_prob_analyser = ProbabilisticSpatialAnalysis(ensemble_selected_prediction[mask_volume], ensemble_predicted_volumes[mask_volume],
                                             anti_selected_prob_dataset, scalers, base_DEM, base_mesh, **temporal_test_dataset_parameters)

In [ ]:
id_dataset = test_dataset_id

test_dataset.graph_files = [test_graph_files[id_dataset]]
# test_dataset.indices = [indices[id_dataset]]
# train_dataset.graph_files = ['dk41_33.pt']
# train_dataset.graph_files = [train_dataset_graph_files[id_dataset]]

rollout_plotter = PlotRollout(plmodel, trainer, test_dataset, 
                              scalers=scalers, type_loss='RMSE', **temporal_test_dataset_parameters)

test_dataset.graph_files = test_graph_files
rollout_plotter._plot_volumes();

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 5))
axs = axs.flatten()

time_vector = get_time_vector(time_stop, temporal_res)

plot_line_with_deviation(time_vector[1:], full_prob_analyser.real_volumes, with_minmax=True, ax=axs[1], c='royalblue')
if anti_prob_analyser.real_volumes.shape[0] > 1:
    plot_line_with_deviation(time_vector[1:], anti_prob_analyser.predicted_volumes, ax=axs[1],with_minmax=True,  label='Predicted ensemble (not plausible)', color="C7")
plot_line_with_deviation(time_vector[1:], prob_analyser.predicted_volumes, with_minmax=True, ax=axs[1], c='C1', label='Predicted ensemble (plausible)')

axs[1].plot(time_vector[1:-1], rollout_plotter.predicted_volume, c='C3', label='Predicted deterministic')

deterministic_vol = test_dataset[test_dataset_id].BC.T * test_dataset[test_dataset_id].edge_BC_length
axs[1].plot(time_vector[1:], np.cumsum(deterministic_vol * temporal_res * 60), c='C2')

axs[1].set_xlim(0, time_vector[-1]*1.05)
# axs[1].legend(loc='upper right', fontsize=15)
axs[1].set_xlabel('Time [h]')
axs[1].set_ylabel('Flood volume [$m^3$]')


plot_line_with_deviation(time_vector[:-1], all_hydrographs[:,:time_stop], with_minmax=True, ax=axs[0], c='royalblue', label='Real ensemble')
axs[0].plot(time_vector[:-1], deterministic_vol, c='C2', label='Real deterministic')
# axs[0].legend()
axs[0].set_xlabel('Time [h]')
axs[0].set_ylabel('Discharge [$m^3$/s]')
# axs[0].set_ylim(0, 400*1.05)

# collect legends from both axes and create a single legend box outside the plot
handles, labels = [], []
for ax in axs:
    h, l = ax.get_legend_handles_labels()
    for hh, ll in zip(h, l):
        if ll not in labels:
            handles.append(hh)
            labels.append(ll)

# place the combined legend below the subplots
fig.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.5, -0.07),
           ncol=min(3, len(labels)), fontsize=14, frameon=True)

panel_labels = ['(a)', '(b)']
axs[0].text(0.98, 0.98, panel_labels[0], transform=axs[0].transAxes,
                fontsize=20, va='top', ha='right')
axs[1].text(0.02, 0.98, panel_labels[1], transform=axs[1].transAxes,
                fontsize=20, va='top', ha='left')

plt.tight_layout(rect=[0, 0.05, 1, 1])  # leave space at bottom for the legend
# plt.savefig(os.path.join(save_folder, 'hydrographs_volumes_ensemble.png'), dpi=300, bbox_inches='tight')

In [ ]:
fig, axs = plot_percentiles(ensemble_selected_prediction, test_dataset=test_dataset[test_dataset_id], quantiles=[0.05, 0.5, 0.95], 
                            variable='WD_max', water_threshold=0.05, vmax=1.5, temporal_res=temporal_res, **prob_analyser.default_plot_kwargs)

fig, axs = plot_WD_max_probability(ensemble_selected_prediction, wd_thresholds=[0.05, 0.3, 1], **prob_analyser.default_plot_kwargs)

In [ ]:
quantiles = [0.05, 0.25, 0.5, 0.75, 0.95]
quantiles = [0.5]
fig, axs = plot_percentile_diff(ensemble_selected_prediction[plausible_runs], test_dataset[test_dataset_id], 
                                deterministic_prediction=rollout_plotter.predicted_WD.map.max(1)[:-1],
                                quantiles=quantiles, variable='WD_max', temporal_res=temporal_res, 
                                diff_value=1, vmax=2.5, **full_prob_analyser.default_plot_kwargs)
# plt.savefig(os.path.join(save_folder, "percentile_diff_quantiles_WD_max.png"))

In [ ]:
# quantiles = [0.05, 0.5, 0.95]
# quantiles = [0.25, 0.5, 0.75]
fig, axs = plot_percentile_diff(ensemble_selected_prediction[plausible_runs], test_dataset[test_dataset_id], 
                                deterministic_prediction=rollout_plotter.pred_FATPlot.map[:-1],
                                quantiles=quantiles, variable='FAT', water_threshold=0.05, temporal_res=temporal_res, 
                                diff_value=96, **full_prob_analyser.default_plot_kwargs)

# plt.savefig(os.path.join(save_folder, "percentile_diff_quantiles_FAT.png"))

In [ ]:
x, y = 175000, 428000

ax = add_inset_distribution_to_ax(prob_analyser.ensemble_selected_prediction[:,:,2], x, y, meshes, ax=None, bins=20)

### Explore single simulation

In [ ]:
id_dataset = 3074

base_data = base_datas[id_dataset//num_breaches]

prob_test_dataset[id_dataset].edge_attr = get_edge_features(base_data, scalers=scalers, **selected_edge_features)
base_data.x = get_node_features(base_data, **selected_node_features, scalers=scalers)
    
WD = replicate_initial_condition(base_data.WD.unsqueeze(-1), previous_t)
V = replicate_initial_condition(base_data.V.unsqueeze(-1) ,previous_t)

prob_test_dataset[id_dataset].x = torch.cat([base_data.x, get_previous_steps(aggregate_WD_V, time_start, previous_t, WD, V)], dim=-1)
prob_test_dataset[id_dataset].y = torch.zeros(1, NUM_WATER_VARS, time_stop, device=device)

del prob_test_dataset[id_dataset].init_WD
# prob_test_dataset[id_dataset].BC *= 2

rollout_plotter = SinglePlotRollout(plmodel, trainer, prob_test_dataset[id_dataset], 
                                    scalers=scalers, type_loss='RMSE', **temporal_test_dataset_parameters)

rollout_plotter.plot_BC();

In [ ]:
rollout_plotter._plot_volumes()
print("Volume R2:", rollout_plotter.get_volumes_r2())
print("Volume ARME:", rollout_plotter.get_volumes_ARME())
# plt.savefig(os.path.join(save_folder, f'example_volumes_{id_dataset:02d}.png'))

In [ ]:
rollout_plotter._mesh_scale_plot(scale=0)

rollout_plotter.predicted_WD.kwargs['vmax'] = 2

plot_times = [1, 2, 3, 4]
plot_times = [1, 5, 10, 30]
plot_times = [5, 10, 20, 30, 45]

rollout_plotter.show_h_rollout(plot_times=plot_times, scale=None, zoom_extent=None)
# rollout_plotter.show_v_rollout(plot_times=plot_times, scale=1, logscale=True, zoom_extent=None)
# rollout_plotter._plot_DEM()
# plt.savefig(f'{save_folder}/area_1_WD_ex.png')

In [ ]:
rollout_plotter._mesh_scale_plot(scale=0)

# rollout_plotter.pred_FATPlot.kwargs['vmax'] = 48

rollout_plotter.show_FAT(water_threshold=0.05, scale=0,
                        #  zoom_extent=rollout_plotter.around_breach
                         )

# plt.savefig(os.path.join(save_folder, f'example_FAT_{id_dataset:02d}.png'))